# Qwen3-VL-8B ChartQA — LoRA 合併、AWQ 量化與 vLLM 效能量測（Colab A100）

**前置**：Phase 2 訓練已完成、LoRA adapter 已在 HF Hub（`<你的帳號>/qwen3vl-8b-chartqa-lora`）。

**用法**（本專案暫不走 GitHub，直接上傳 notebook）：
1. Colab → 檔案 → 上傳筆記本 → 選這個檔案；執行階段選 **A100 GPU**
2. 左側 🔑 Secrets 確認 `HF_TOKEN`（write token）且「筆記本存取權」已開啟
3. 本 notebook 分四個 stage，**建議分四次執行**（unsloth / llm-compressor / vLLM 依賴互相衝突；offline eval 後也重啟，避免 GPU 記憶體碎片影響 serving benchmark）：

| 執行 | 開啟的 flag | 做什麼 | 之後 |
|---|---|---|---|
| 第 1 次 | `DO_MERGE=True`（其餘 False） | LoRA 合併 → merged-16bit push | 執行階段 → 重新啟動 |
| 第 2 次 | `DO_QUANT=True`（其餘 False） | AWQ W4A16 g32 量化 → push | 執行階段 → 重新啟動 |
| 第 3 次 | `DO_EVAL_QUANT=True`（其餘 False） | 量化 sanity check | 執行階段 → 重新啟動 |
| 第 4 次 | `DO_BENCH=True`（其餘 False） | vLLM serving benchmark | 中斷連線並刪除執行階段 |

4. 每個 stage 的輸入/輸出都在 HF Hub —— 斷線後重開只要開對 flag 再「全部執行」即可續跑
5. 新流程先 `SMOKE_TEST = True` 快跑再改 `False`；本專案目前 smoke 已完成，預設直接接正式 256 筆校準

**產物**（自動 push 到 HF Hub）：
- `<你的帳號>/qwen3vl-8b-chartqa-merged-16bit` — 合併後 16-bit 模型（~17.5GB）
- `<你的帳號>/qwen3vl-8b-chartqa-awq` — AWQ 量化模型（~7GB）+ `eval_quant/`（量化前後對照）+ `bench/`（效能量測表）


In [ ]:
# 1. GPU 檢查（本 notebook 需要 A100 40GB）
# 注意：這裡刻意不 import torch —— vLLM 安裝（cell 3）會替換 torch，必須發生在任何 torch import 之前
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
IS_A100 = "A100" in gpu
print(f"GPU: {gpu}")
if not IS_A100:
    print("!! 警告：非 A100 —— 需要載 merged-16bit 的 stage（1/3/4）幾乎必定 OOM")

In [ ]:
# 2. Stage 開關 —— 放在安裝格之前，安裝按需進行（四次執行建議見最上方表格）
DO_MERGE      = False   # Stage 1：已完成；LoRA 合併 -> merged-16bit push
DO_QUANT      = True    # <<< 下一步：Stage 2 正式 AWQ W4A16 g32 量化 -> awq push
DO_EVAL_QUANT = False   # Stage 3：量化 sanity check（merged vs awq，小樣本 relaxed accuracy）
DO_BENCH      = False   # Stage 4：vLLM serving benchmark（TTFT/TPOT/吞吐）
BENCH_16BIT   = True    # Stage 4 正式結果加測 merged-16bit，和 AWQ 做效能對照
SMOKE_TEST    = False   # <<< 現有 AWQ 已通過 smoke；重跑 256 筆正式校準並保存 metadata

# 不同 stage 的套件互相衝突，不要同 session 混跑（重啟 runtime 後改 flag 再跑）
assert not (DO_MERGE and DO_QUANT), "DO_MERGE 與 DO_QUANT 要分開兩次執行"
assert not (DO_MERGE and (DO_EVAL_QUANT or DO_BENCH)), "unsloth 與 vLLM 不要同 session"
assert not (DO_QUANT and (DO_EVAL_QUANT or DO_BENCH)), "llm-compressor 與 vLLM 不要同 session"
print(f"{DO_MERGE=} {DO_QUANT=} {DO_EVAL_QUANT=} {DO_BENCH=} {BENCH_16BIT=} {SMOKE_TEST=}")

In [ ]:
%%capture
# 3. 條件安裝 —— 只裝當前 stage 需要的
#    vLLM 會安裝它自己 pin 的 torch 蓋掉 Colab 預裝版，所以本格之前不能 import torch
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"  # 加速下載器在 Colab 上會靜默卡死（train/eval notebook 踩過的同一個問題）
os.environ["HF_HUB_DISABLE_XET"] = "1"  # 同上，Xet 傳輸在 Colab/GCP 有已知卡死問題
!pip install hf_transfer "huggingface_hub>=0.34.0"

if DO_MERGE:
    # unsloth 官方 Colab 安裝區塊（與訓練 notebook 相同）+ 版本 pin
    import re
    if "COLAB_" not in "".join(os.environ.keys()):
        !pip install unsloth
    else:
        import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
        xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
        !pip install sentencepiece protobuf "datasets==4.3.0" hf_transfer
        !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
        !pip install --no-deps --upgrade "torchao>=0.16.0"
    !pip install transformers==4.57.1
    !pip install --no-deps trl==0.22.2

if DO_QUANT:
    # 0.12.0：AWQModifier 移到 modifiers.transform.awq、量化設定改放 QuantizationModifier（cell 11）
    # 不釘 transformers/datasets 版本 —— llmcompressor 0.12.0 需要 transformers>=5.9.0、
    # datasets>=4.8.4，跟其他 stage 沿用的 4.57.1/4.3.0（unsloth 需要）互斥；
    # 讓 pip 自己解出相容版本（Dataset.from_parquet 是穩定舊 API，不挑版本）
    !pip install "llmcompressor==0.12.0"

if DO_EVAL_QUANT or DO_BENCH:
    !pip install -U vllm
    !pip install httpx datasets

!pip uninstall -y -q hf_xet  # 雙保險：直接移除套件，避免 huggingface_hub 選用 Xet 傳輸而卡死

In [ ]:
# 4. 從 Colab Secrets 讀 HF_TOKEN 登入
from google.colab import userdata
from huggingface_hub import HfApi, login, whoami
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)
HF_USER = whoami()["name"]
api = HfApi()
print("logged in as:", HF_USER)

def _retry_hf(fn, *args, max_retries=5, base_delay=10.0, **kwargs):
    # HF Hub 的 Xet CDN 簽章在 Colab 上會間歇性失效，表現方式很多種（403、被包裝成
    # 「檔案不存在」的 OSError、離線載入殘缺快取的 AttributeError）。這裡單純重試；
    # 不能塞 force_download —— unsloth 內部是「先預下載、再離線載入」，離線階段
    # 收到 force_download 會直接 ValueError
    import time

    for attempt in range(1, max_retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            if attempt == max_retries:
                raise
            wait = base_delay * attempt
            print(f"  [HF retry] attempt {attempt}/{max_retries} failed ({type(e).__name__}: {e}); retrying in {wait:.0f}s")
            time.sleep(wait)

def _prefetch_repo(repo_id, repo_type="model", max_retries=10, base_delay=15.0):
    # 真正的關鍵：先用整檔下載把 repo 抓齊進本機快取（走已證實可靠的路徑，失敗自動重試）。
    # unsloth/transformers 的「預下載失敗 -> 退回離線載入」流程在快取不完整時會炸出
    # 各種怪錯（checkpoint_files None / 檔案不存在）；快取抓齊之後離線載入必定成功
    import time
    from huggingface_hub import snapshot_download

    for attempt in range(1, max_retries + 1):
        try:
            return snapshot_download(repo_id, repo_type=repo_type)
        except Exception as e:
            if attempt == max_retries:
                raise
            wait = base_delay * attempt
            print(f"  [prefetch {repo_id}] attempt {attempt}/{max_retries} failed ({type(e).__name__}); retrying in {wait:.0f}s")
            time.sleep(wait)

def _prefetch_model_and_base(repo_id):
    # LoRA adapter repo 會連同 adapter_config.json 指到的底模一起預抓
    import json, os
    path = _prefetch_repo(repo_id)
    cfg = os.path.join(path, "adapter_config.json")
    if os.path.exists(cfg):
        base = json.load(open(cfg)).get("base_model_name_or_path")
        if base:
            print(f"[prefetch] adapter 底模: {base}")
            _prefetch_repo(base)
    return path

In [ ]:
# 5. 設定
SEED = 3407

ADAPTER_REPO = f"{HF_USER}/qwen3vl-8b-chartqa-lora"          # Phase 2 產物（輸入）
MERGED_REPO  = f"{HF_USER}/qwen3vl-8b-chartqa-merged-16bit"  # Stage 1 產物
AWQ_REPO     = f"{HF_USER}/qwen3vl-8b-chartqa-awq"           # Stage 2 產物 + eval_quant/ + bench/

# Stage 2：AWQ 校準（PLAN：W4A16、group_size=32；校準用 ChartQA 自家資料，領域對齊）
CALIB_SAMPLES = 16 if SMOKE_TEST else 256
CALIB_MAX_LEN = 2048

# Stage 3：量化 sanity check（human/augmented 各 N 筆）
EVAL_QUANT_N   = 20 if SMOKE_TEST else 100
MAX_NEW_TOKENS = 32

# Stage 3/4 共用：vLLM
VLLM_MAX_MODEL_LEN = 8192
VLLM_GPU_UTIL      = 0.90

# Stage 4：benchmark
CONCURRENCY_LEVELS       = [1] if SMOKE_TEST else [1, 4, 8, 16]
BENCH_REQUESTS_PER_LEVEL = 8 if SMOKE_TEST else 32
BENCH_MAX_TOKENS         = 64
BENCH_WARMUP             = 2   # 每個並發等級前的暖身請求數（不計分）

print(f"{ADAPTER_REPO=}\n{MERGED_REPO=}\n{AWQ_REPO=}\n{SMOKE_TEST=}")

In [ ]:
# 6. ChartQA 載入 + prompt + relaxed accuracy（與 src/chartqa_data.py、src/relaxed_accuracy.py 保持同步）
#    以及圖片 -> base64 data URI（Stage 3/4 的 vLLM 推論用 OpenAI 訊息格式）
import base64, io
from datasets import Dataset

DATASET_ID = "HuggingFaceM4/ChartQA"
ANSWER_INSTRUCTION = "Answer the question using a single word or phrase."
HUMAN, MACHINE = 0, 1

def _download_with_retry(filename, max_retries=6, base_delay=5.0):
    # HF 的 Xet CDN 簽章在 Colab 上會間歇性失效（同一檔案這次成功、下次 403），
    # 不是檔案本身壞掉；重新請求會拿到新的簽章 URL，重試就會過
    import time
    from huggingface_hub import hf_hub_download
    from huggingface_hub.errors import HfHubHTTPError

    for attempt in range(1, max_retries + 1):
        try:
            return hf_hub_download(DATASET_ID, filename, repo_type="dataset")
        except HfHubHTTPError as e:
            if attempt == max_retries:
                raise
            wait = base_delay * attempt
            print(f"  [{filename}] attempt {attempt}/{max_retries} failed ({type(e).__name__}); retrying in {wait:.0f}s")
            time.sleep(wait)

def load_chartqa(split, n=None, human_or_machine=None, seed=42):
    # 直接抓該 split 的 parquet 檔（整檔循序下載＋失敗重試），不用 datasets.load_dataset()：
    # 它的 eager 模式會連其他 split 一起準備，streaming 模式則用 byte-range 讀取，
    # 都撞過 HF Xet CDN 在 Colab 上簽章失效的 403（結果證實是間歇性的，不是特定檔案壞掉）
    from huggingface_hub import HfApi

    files = sorted(f for f in HfApi().list_repo_files(DATASET_ID, repo_type="dataset")
                   if f.startswith(f"data/{split}-"))
    local_paths = [_download_with_retry(f) for f in files]
    ds = Dataset.from_parquet(local_paths)
    if human_or_machine is not None:
        ds = ds.filter(lambda ex: ex["human_or_machine"] == human_or_machine)
    if n is not None and n < len(ds):
        ds = ds.shuffle(seed=seed).select(range(n))
    return ds

def get_answer(example):
    label = example["label"]
    return str(label[0]) if isinstance(label, list) else str(label)

def to_messages(example, include_answer=True):
    user_content = [
        {"type": "image", "image": example["image"]},
        {"type": "text", "text": f"{example['query']}\n{ANSWER_INSTRUCTION}"},
    ]
    messages = [{"role": "user", "content": user_content}]
    if include_answer:
        messages.append({"role": "assistant",
                         "content": [{"type": "text", "text": get_answer(example)}]})
    return messages

def _to_float(text):
    try:
        if text.endswith("%"):
            return float(text.rstrip("%")) / 100.0
        return float(text)
    except ValueError:
        return None

def relaxed_correctness(prediction, target, max_relative_change=0.05):
    prediction_float = _to_float(prediction)
    target_float = _to_float(target)
    if prediction_float is not None and target_float:
        relative_change = abs(prediction_float - target_float) / abs(target_float)
        return relative_change <= max_relative_change
    return prediction.lower() == target.lower()

def normalize_prediction(text):
    text = text.strip()
    if text.endswith("."):
        text = text[:-1].rstrip()
    return text

def relaxed_accuracy(predictions, targets):
    assert len(predictions) == len(targets)
    if not predictions:
        return 0.0
    return sum(relaxed_correctness(normalize_prediction(p), t)
               for p, t in zip(predictions, targets)) / len(predictions)

def to_data_uri(img, max_side=1024):
    # 統一縮到 max_side、JPEG q90：固定視覺 token 數，benchmark 才有可比性
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

def chat_messages(ex):
    # OpenAI 訊息格式：vLLM offline 的 llm.chat 與 serving 的 /v1/chat/completions 通用
    return [{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": to_data_uri(ex["image"])}},
        {"type": "text", "text": f"{ex['query']}\n{ANSWER_INSTRUCTION}"},
    ]}]

## Stage 1：LoRA 合併 → merged-16bit（`DO_MERGE`）

**路線**：unsloth `push_to_hub_merged(save_method="merged_16bit")`，直接從 adapter repo 載入（unsloth 會自動抓 4-bit 底模並掛上 adapter）。LoRA delta 是相對 **nf4 反量化權重**學出來的，unsloth 的 merged_16bit 把同一份底模 upcast 成 16-bit 再疊 delta —— 合併結果最忠實重現 Phase 3 已量測的行為。

**Fallback（僅在上面失敗時用）**：下下格的 peft 路線 —— 把 adapter 疊到 `Qwen/Qwen3-VL-8B-Instruct` 官方 bf16 權重後 `merge_and_unload()`。該底模與訓練時看到的 nf4 反量化權重略有差異，delta 疊上去會小幅漂移，故僅作備援（此差異會被下一步 AWQ 的量化誤差稀釋）。

In [ ]:
# 7. Stage 1：合併並 push（約 15-25 分鐘，多數時間在上傳 ~17.5GB）
if DO_MERGE:
    from unsloth import FastVisionModel
    _prefetch_model_and_base(ADAPTER_REPO)   # 先抓齊檔案，unsloth 離線載入才不會踩到殘缺快取
    _prefetch_repo("unsloth/Qwen3-VL-8B-Instruct")  # push_to_hub_merged 內部另外需要這份全精度版本做合併，不在上面的清單裡
    model, tokenizer = _retry_hf(FastVisionModel.from_pretrained, ADAPTER_REPO, load_in_4bit=True)
    model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method="merged_16bit", token=HF_TOKEN)

    # 驗證：merged repo 至少要有 safetensors 分片 + config
    files = api.list_repo_files(MERGED_REPO)
    assert any(f.endswith(".safetensors") for f in files), files
    assert "config.json" in files, files
    print(f"repo files: {len(files)}  safetensors shards: {sum(f.endswith('.safetensors') for f in files)}")
    print(f"done -> https://huggingface.co/{MERGED_REPO}")
else:
    print("skip（DO_MERGE=False）")

In [ ]:
# 8. Fallback：peft 手動合併 —— 僅在上一格失敗時把 RUN_PEFT_FALLBACK 改 True 執行（CPU 合併，不吃 VRAM）
RUN_PEFT_FALLBACK = False
if RUN_PEFT_FALLBACK:
    import torch
    from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
    from peft import PeftModel
    _prefetch_repo("Qwen/Qwen3-VL-8B-Instruct")
    _prefetch_repo(ADAPTER_REPO)
    base = _retry_hf(Qwen3VLForConditionalGeneration.from_pretrained,
        "Qwen/Qwen3-VL-8B-Instruct", dtype=torch.bfloat16)   # 不給 device_map -> CPU
    merged = PeftModel.from_pretrained(base, ADAPTER_REPO).merge_and_unload()
    merged.push_to_hub(MERGED_REPO, token=HF_TOKEN)
    AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct").push_to_hub(MERGED_REPO, token=HF_TOKEN)
    print(f"done (peft fallback) -> https://huggingface.co/{MERGED_REPO}")

In [ ]:
# 9. 釋放記憶體（Stage 1 結束）
import gc
for _n in ("model", "tokenizer", "base", "merged"):
    globals().pop(_n, None)
gc.collect()
try:
    import torch
    torch.cuda.empty_cache()
    print(f"VRAM allocated: {torch.cuda.memory_allocated()/1024**3:.1f} GB")
except Exception:
    pass
if DO_MERGE:
    print(">>> Stage 1 完成。執行階段 → 重新啟動，改開 DO_QUANT 後再「全部執行」")

## Stage 2：AWQ W4A16 g32 量化（`DO_QUANT`）

- **工具**：`llmcompressor==0.12.0`。0.12 起 `AWQModifier` 移到 `llmcompressor.modifiers.transform.awq` 且只負責 AWQ scale 搜尋；量化設定（bits / group_size / ignore）放在 `QuantizationModifier`，recipe 是兩元素 list（官方 Qwen3-VL 範例同此寫法）
- **不量化**：vision tower（`visual.*`，含 merger/projector）與 `lm_head` —— 佔總權重僅 1-3%，量化 ROI 低且易傷視覺品質（PLAN 查證的業界標準做法）
- **group_size=32**：PLAN 指定；config_groups 寫法照官方 Qwen3-VL MoE AWQ 範例（int4、symmetric、observer=mse）
- **校準集**：ChartQA train 抽 256 筆（SMOKE 16 筆），走與訓練一致的 chat template —— 領域對齊比官方範例的 flickr30k 更貼近部署分佈
- 模型載到 **CPU**，llm-compressor 預設 sequential onloading 逐層搬上 GPU 校準，40GB 十分充裕
- 輸出為 **compressed-tensors 格式**（非 AutoAWQ 格式）：vLLM 自動偵測、免 `--quantization` 旗標；log 顯示 `compressed-tensors` 而非 `awq` 是預期行為

In [ ]:
# 10. Stage 2：載入 merged 模型（CPU）+ ChartQA 多模態校準集
if DO_QUANT:
    import torch
    from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

    _prefetch_repo(MERGED_REPO)
    model = _retry_hf(Qwen3VLForConditionalGeneration.from_pretrained, MERGED_REPO, dtype=torch.bfloat16)  # 不給 device_map -> CPU
    processor = _retry_hf(AutoProcessor.from_pretrained, MERGED_REPO)

    calib_raw = load_chartqa("train", n=CALIB_SAMPLES, seed=SEED)

    def preprocess_and_tokenize(example):
        msgs = to_messages(example, include_answer=True)   # 含答案，與訓練分佈一致
        text = processor.apply_chat_template(msgs, tokenize=False)
        return processor(text=[text], images=[example["image"].convert("RGB")],
                         padding=False, max_length=CALIB_MAX_LEN, truncation=True)

    calib_ds = calib_raw.map(preprocess_and_tokenize, remove_columns=calib_raw.column_names)

    def data_collator(batch):
        assert len(batch) == 1
        return {key: torch.tensor(value) for key, value in batch[0].items()}

    print(f"calibration rows: {len(calib_ds)}")

In [ ]:
# 11. AWQ 量化（sequential onloading 逐層校準；SMOKE 約 10 分鐘、256 筆約 30-60 分鐘）
if DO_QUANT:
    from llmcompressor import oneshot
    from llmcompressor.modifiers.quantization import QuantizationModifier
    from llmcompressor.modifiers.transform.awq import AWQModifier

    recipe = [
        AWQModifier(duo_scaling=False),   # 官方 Qwen3-VL dense 範例設定
        QuantizationModifier(
            ignore=["re:.*lm_head", "re:.*visual.*", "re:model[.]visual.*"],
            config_groups={"group_0": {
                "targets": ["Linear"],
                "weights": {"num_bits": 4, "type": "int", "symmetric": True,
                            "strategy": "group", "group_size": 32,
                            "dynamic": False, "observer": "mse"},
            }},
        ),
    ]

    # 若 sequential tracing 報錯：改加 pipeline="basic"（較耗 VRAM 但不用逐層追蹤）
    oneshot(
        model=model,
        dataset=calib_ds,
        recipe=recipe,
        max_seq_length=CALIB_MAX_LEN,
        num_calibration_samples=len(calib_ds),
        data_collator=data_collator,
        sequential_targets=["Qwen3VLTextDecoderLayer"],   # 官方範例：沿 text decoder 逐層校準
    )

In [ ]:
# 12. 驗證：vision tower / lm_head 未量化、language Linear 已量化 g32；量化後煙霧生成 1 題
if DO_QUANT:
    quantized = [n for n, m in model.named_modules()
                 if getattr(m, "quantization_scheme", None) is not None]
    bad = [n for n in quantized if "visual" in n or "lm_head" in n]
    assert quantized, "沒有任何模組被量化？"
    assert not bad, f"這些模組不該被量化: {bad[:5]}"
    scheme = next(m.quantization_scheme for _, m in model.named_modules()
                  if getattr(m, "quantization_scheme", None) is not None)
    print(f"quantized modules: {len(quantized)}（vision/lm_head 均未量化）")
    print("sample weight scheme:", scheme.weights)

    from compressed_tensors.offload import dispatch_model
    dispatch_model(model)   # 校準後模型在 CPU，搬上 GPU 做煙霧生成
    ex = load_chartqa("val", n=1, seed=SEED)[0]
    msgs = to_messages(ex, include_answer=False)
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[ex["image"].convert("RGB")],
                       return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=32, do_sample=False)
    pred = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                  skip_special_tokens=True)[0]
    print(f"Q: {ex['query']}\n  gold: {get_answer(ex)}\n  pred: {pred.strip()}")

In [ ]:
# 13. 存檔（compressed-tensors 格式）+ 可重現性 metadata + push 到 AWQ_REPO
if DO_QUANT:
    SAVE_DIR = "qwen3vl-8b-chartqa-awq"
    model.save_pretrained(SAVE_DIR, save_compressed=True)
    processor.save_pretrained(SAVE_DIR)

    # recipe.yaml 不會記錄校準樣本數；另存 metadata 才能區分 smoke 與正式權重
    import importlib.metadata, json, platform, torch
    from datetime import datetime, timezone
    def _version(name):
        try:
            return importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            return None
    quant_metadata = {
        "schema_version": 1,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_model": MERGED_REPO,
        "adapter": ADAPTER_REPO,
        "method": "AWQ / compressed-tensors W4A16",
        "weight_bits": 4, "group_size": 32,
        "ignored_modules": ["re:.*lm_head", "re:.*visual.*", "re:model[.]visual.*"],
        "calibration_dataset": "HuggingFaceM4/ChartQA train",
        "calibration_samples": CALIB_SAMPLES,
        "calibration_max_length": CALIB_MAX_LEN,
        "seed": SEED, "smoke_test": SMOKE_TEST,
        "gpu": torch.cuda.get_device_name(0),
        "python": platform.python_version(),
        "packages": {n: _version(n) for n in
                     ["torch", "transformers", "llmcompressor", "compressed-tensors", "datasets"]},
    }
    with open(f"{SAVE_DIR}/quantization_metadata.json", "w") as f:
        json.dump(quant_metadata, f, indent=2, ensure_ascii=False)
    api.create_repo(AWQ_REPO, exist_ok=True)
    api.upload_folder(folder_path=SAVE_DIR, repo_id=AWQ_REPO)
    print(f"done -> https://huggingface.co/{AWQ_REPO}")

    import gc, torch
    globals().pop("model", None)
    gc.collect(); torch.cuda.empty_cache()
    print(">>> Stage 2 完成。執行階段 → 重新啟動，只開 DO_EVAL_QUANT 後再「全部執行」")

## Stage 3：量化 sanity check（`DO_EVAL_QUANT`）— merged-16bit vs AWQ

小樣本 relaxed accuracy 對照，確認量化沒有明顯掉分（PLAN Verification：量化前後要在小樣本跑一次）。

> 注意：本 stage 用 **vLLM** 推論，數字與 Phase 3 評估（unsloth + bnb-4bit）**不直接可比** —— 只看「量化前 vs 量化後」的差距。每個模型跑完會先把預測 push 到 Hub 的 `eval_quant/`，斷線重跑會直接重用已完成的部分。

In [ ]:
# 14. Stage 3：vLLM offline 推論 + relaxed accuracy（同一函式跑兩個模型）
if DO_EVAL_QUANT:
    import gc, json, os, torch
    from huggingface_hub import hf_hub_download
    from vllm import LLM, SamplingParams

    os.makedirs("eval_quant", exist_ok=True)

    def eval_with_vllm(repo, tag):
        # 斷線續跑：Hub 上已有這個 tag 的預測就直接重用
        try:
            path = hf_hub_download(AWQ_REPO, f"eval_quant/preds_{tag}_n{EVAL_QUANT_N}.json")
            print(f"[{tag}] 重用 Hub 上既有預測（要重算請先刪掉該檔）")
            return json.load(open(path))
        except Exception:
            pass

        _prefetch_repo(repo)   # 先抓齊，vLLM 載入途中才不會撞 CDN 403
        llm = LLM(model=repo, max_model_len=VLLM_MAX_MODEL_LEN,
                  gpu_memory_utilization=VLLM_GPU_UTIL,
                  limit_mm_per_prompt={"image": 1})
        sp = SamplingParams(temperature=0, max_tokens=MAX_NEW_TOKENS)
        results = {}
        for name, hom in [("human", HUMAN), ("augmented", MACHINE)]:
            ds = load_chartqa("test", n=EVAL_QUANT_N, human_or_machine=hom, seed=SEED)
            outs = llm.chat([chat_messages(ex) for ex in ds], sp)
            preds = [o.outputs[0].text.strip() for o in outs]
            golds = [get_answer(ex) for ex in ds]
            acc = relaxed_accuracy(preds, golds)
            results[name] = {"n": len(ds), "relaxed_accuracy": acc,
                             "predictions": preds, "golds": golds}
            print(f"[{tag}] {name}: relaxed_accuracy = {acc:.4f}  (n={len(ds)})")

        with open(f"eval_quant/preds_{tag}_n{EVAL_QUANT_N}.json", "w") as f:
            json.dump(results, f, indent=1)
        api.create_repo(AWQ_REPO, exist_ok=True)
        api.upload_file(path_or_fileobj=f"eval_quant/preds_{tag}_n{EVAL_QUANT_N}.json",
                        path_in_repo=f"eval_quant/preds_{tag}_n{EVAL_QUANT_N}.json", repo_id=AWQ_REPO)

        # 同 session 還要載第二個模型 -> 先釋放引擎
        # （若下一個 LLM() 仍 OOM：重啟 runtime、只開 DO_EVAL_QUANT 重跑，已完成的 tag 會自動重用）
        del llm
        gc.collect(); torch.cuda.empty_cache()
        return results

In [ ]:
# 15. 跑 merged-16bit 與 AWQ -> 對照表 -> eval_quant/results.json push
if DO_EVAL_QUANT:
    import pandas as pd

    res16 = eval_with_vllm(MERGED_REPO, "merged16")
    res4  = eval_with_vllm(AWQ_REPO, "awq")

    rows = []
    for split in ["human", "augmented"]:
        rows.append({
            "test split": split,
            "n": res16[split]["n"],
            "merged-16bit": round(res16[split]["relaxed_accuracy"], 4),
            "awq-w4a16-g32": round(res4[split]["relaxed_accuracy"], 4),
            "Δ": round(res4[split]["relaxed_accuracy"] - res16[split]["relaxed_accuracy"], 4),
        })
    n_tot = sum(res16[s]["n"] for s in res16)
    ov16 = sum(res16[s]["relaxed_accuracy"] * res16[s]["n"] for s in res16) / n_tot
    ov4  = sum(res4[s]["relaxed_accuracy"] * res4[s]["n"] for s in res4) / n_tot
    rows.append({"test split": "overall", "n": n_tot,
                 "merged-16bit": round(ov16, 4), "awq-w4a16-g32": round(ov4, 4),
                 "Δ": round(ov4 - ov16, 4)})
    df = pd.DataFrame(rows)
    print(df.to_markdown(index=False))

    if ov16 - ov4 > 0.02:
        print(f"!! 量化掉分 {(ov16-ov4)*100:.1f}pp > 2pp —— "
              f"考慮把首末 decoder layer 或 re:.*down_proj 加進 ignore 重跑 Stage 2")

    import json
    summary = {"merged": MERGED_REPO, "awq": AWQ_REPO, "eval_n_per_split": EVAL_QUANT_N,
               "metric": "relaxed_accuracy(5%)", "engine": "vllm", "table": rows}
    with open("eval_quant/results.json", "w") as f:
        json.dump(summary, f, indent=1)
    api.upload_file(path_or_fileobj="eval_quant/results.json",
                    path_in_repo="eval_quant/results.json", repo_id=AWQ_REPO)
    print(f"pushed -> https://huggingface.co/{AWQ_REPO}/tree/main/eval_quant")

## Stage 4：vLLM serving benchmark（`DO_BENCH`）

- **指標**：TTFT（送出請求到第一個 token）、TPOT（首 token 後平均每 token 時間 = (總時長−TTFT)/(tokens−1)）、output tokens/s、requests/s；報 p50 / p95
- **負載**：ChartQA test 真實圖表（統一縮到 ≤1024、JPEG q90 固定視覺 token 數）、`temperature=0`、`max_tokens=64`、並發 1/4/8/16
- 每個並發等級先丟 `BENCH_WARMUP` 筆暖身不計分；`BENCH_16BIT=True` 會再起 merged-16bit server 跑一輪對照（40GB 放得下，但 KV cache 空間小很多）
- server 由 notebook 背景啟動（subprocess），輪詢 `/health` 就緒後才開跑；量測 cell 直接 `await`（Colab 支援 top-level await）

In [ ]:
# 16. Stage 4：vLLM server 啟停（背景 subprocess + /health 輪詢）
if DO_BENCH:
    import os, signal, subprocess, time
    import httpx

    def start_vllm(repo, port=8000, log_path="vllm.log"):
        os.environ.setdefault("OMP_NUM_THREADS", "1")  # Qwen3-VL 官方建議，避免前處理 CPU contention
        cmd = ["vllm", "serve", repo,
               "--port", str(port),
               "--max-model-len", str(VLLM_MAX_MODEL_LEN),
               "--gpu-memory-utilization", str(VLLM_GPU_UTIL),
               "--limit-mm-per-prompt.image", "1",
               "--limit-mm-per-prompt.video", "0",
               "--async-scheduling"]
        proc = subprocess.Popen(cmd, stdout=open(log_path, "w"),
                                stderr=subprocess.STDOUT, preexec_fn=os.setsid)
        deadline = time.time() + 15 * 60   # 首次載模 + 編譯可達 10 分鐘
        while time.time() < deadline:
            if proc.poll() is not None:
                print(open(log_path).read()[-3000:])
                raise RuntimeError("vLLM 啟動即退出，log 尾段如上")
            try:
                if httpx.get(f"http://localhost:{port}/health", timeout=2).status_code == 200:
                    print(f"vLLM ready: {repo}")
                    return proc
            except Exception:
                pass
            time.sleep(5)
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)   # 逾時：別留下佔著 VRAM 的殭屍 server
        raise TimeoutError("等不到 /health —— 看 vllm.log")

    def stop_vllm(proc):
        if proc.poll() is None:
            os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        try:
            proc.wait(timeout=120)
        except subprocess.TimeoutExpired:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
            proc.wait(timeout=30)
        time.sleep(5)   # 等 VRAM 完全釋放

In [ ]:
# 17. async benchmark client（httpx streaming；量 TTFT / TPOT / 吞吐）
if DO_BENCH:
    import asyncio, json, time
    import numpy as np

    async def one_request(client, payload):
        t0 = time.perf_counter(); ttft = None; n_tokens = 0
        async with client.stream("POST", "/v1/chat/completions", json=payload) as r:
            r.raise_for_status()
            async for line in r.aiter_lines():
                if not line.startswith("data: ") or line.strip() == "data: [DONE]":
                    continue
                chunk = json.loads(line[len("data: "):])
                if chunk.get("usage"):                     # include_usage：最後一個 chunk
                    n_tokens = chunk["usage"]["completion_tokens"]
                elif chunk.get("choices") and chunk["choices"][0]["delta"].get("content"):
                    if ttft is None:
                        ttft = time.perf_counter() - t0
        total = time.perf_counter() - t0
        tpot = (total - ttft) / max(n_tokens - 1, 1) if ttft is not None else None
        return {"ttft": ttft, "total": total, "tokens": n_tokens, "tpot": tpot}

    async def bench_level(payloads, concurrency, port=8000):
        sem = asyncio.Semaphore(concurrency)
        async with httpx.AsyncClient(base_url=f"http://localhost:{port}",
                                     timeout=300) as client:
            for p in payloads[:BENCH_WARMUP]:              # 暖身，不計分
                await one_request(client, p)
            async def guarded(p):
                async with sem:
                    return await one_request(client, p)
            t0 = time.perf_counter()
            results = await asyncio.gather(*[guarded(p) for p in payloads[BENCH_WARMUP:]])
            wall = time.perf_counter() - t0
        ttfts = [r["ttft"] for r in results if r["ttft"] is not None]
        tpots = [r["tpot"] for r in results if r["tpot"] is not None]
        toks = sum(r["tokens"] for r in results)
        pct = lambda a, q: round(float(np.percentile(a, q)) * 1000, 1)   # s -> ms
        return {"concurrency": concurrency, "n_requests": len(results),
                "wall_s": round(wall, 2),
                "requests_per_s": round(len(results) / wall, 3),
                "output_tokens_per_s": round(toks / wall, 1),
                "ttft_p50_ms": pct(ttfts, 50), "ttft_p95_ms": pct(ttfts, 95),
                "tpot_p50_ms": pct(tpots, 50), "tpot_p95_ms": pct(tpots, 95)}

In [ ]:
# 18. 跑 benchmark（AWQ；BENCH_16BIT=True 再加 merged-16bit 對照）
if DO_BENCH:
    import pandas as pd

    bench_examples = load_chartqa("test", n=BENCH_WARMUP + BENCH_REQUESTS_PER_LEVEL, seed=SEED)
    bench_msgs = [chat_messages(ex) for ex in bench_examples]   # data URI 先算好，不佔量測時間
    print(f"requests/level: {BENCH_REQUESTS_PER_LEVEL} (+{BENCH_WARMUP} warmup), levels: {CONCURRENCY_LEVELS}")

    targets = [("awq-w4a16-g32", AWQ_REPO)]
    if BENCH_16BIT:
        targets.append(("merged-16bit", MERGED_REPO))

    all_rows = []
    for tag, repo in targets:
        _prefetch_repo(repo)
        proc = start_vllm(repo)
        try:
            payloads = [{"model": repo, "messages": m, "temperature": 0,
                         "max_tokens": BENCH_MAX_TOKENS, "stream": True,
                         "stream_options": {"include_usage": True}} for m in bench_msgs]
            for c in CONCURRENCY_LEVELS:
                row = {"model": tag, **(await bench_level(payloads, c))}
                all_rows.append(row)
                print(row)
        finally:
            stop_vllm(proc)

    df_bench = pd.DataFrame(all_rows)
    print(df_bench.to_markdown(index=False))

In [ ]:
# 19. benchmark 結果表 + 圖 -> bench/ push 到 AWQ_REPO
if DO_BENCH:
    import json, os
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    os.makedirs("bench", exist_ok=True)
    import importlib.metadata, platform, torch
    def _version(name):
        try:
            return importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            return None
    config = {"levels": CONCURRENCY_LEVELS, "requests_per_level": BENCH_REQUESTS_PER_LEVEL,
              "warmup": BENCH_WARMUP, "max_tokens": BENCH_MAX_TOKENS,
              "max_model_len": VLLM_MAX_MODEL_LEN, "image_max_side": 1024,
              "dataset": "ChartQA test", "smoke_test": SMOKE_TEST,
              "gpu": torch.cuda.get_device_name(0), "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
              "gpu_memory_utilization": VLLM_GPU_UTIL, "async_scheduling": True,
              "omp_num_threads": os.environ.get("OMP_NUM_THREADS", "1"),
              "python": platform.python_version(),
              "packages": {n: _version(n) for n in ["vllm", "torch", "transformers"]}}
    with open("bench/benchmark_results.json", "w") as f:
        json.dump({"config": config, "rows": all_rows}, f, indent=1)
    with open("bench/benchmark_table.md", "w") as f:
        f.write(df_bench.to_markdown(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for tag in df_bench["model"].unique():
        sub = df_bench[df_bench["model"] == tag]
        axes[0].plot(sub["concurrency"], sub["output_tokens_per_s"], marker="o", label=tag)
        axes[1].plot(sub["concurrency"], sub["ttft_p50_ms"], marker="o", label=tag)
    axes[0].set(xlabel="concurrency", ylabel="output tokens/s", title="Throughput")
    axes[1].set(xlabel="concurrency", ylabel="TTFT p50 (ms)", title="Latency")
    for ax_ in axes:
        ax_.grid(alpha=0.3); ax_.legend()
    fig.tight_layout()
    fig.savefig("bench/latency_throughput.png", dpi=150)

    api.create_repo(AWQ_REPO, exist_ok=True)
    api.upload_folder(folder_path="bench", path_in_repo="bench", repo_id=AWQ_REPO)
    print(f"pushed -> https://huggingface.co/{AWQ_REPO}/tree/main/bench")
    plt.show()

## 下一步

- `SMOKE_TEST=True` 只是流程驗證；**正式數字要 `SMOKE_TEST=False` 重跑**（校準 256、評估各 100、並發 1/4/8/16 × 32 筆）
- 跑完記錄 cell 15 的量化對照表與 cell 18 的 benchmark 表
- 本機執行 `uv run python scripts/sync_assets_from_hub.py --with-bench` 把 eval / bench 產物拉進 `assets/`
- 接 Phase 5（Colab Gradio demo；目前本機 RTX 2050 4GB 不執行模型 inference）與 Phase 6（HF Hub / Space 上架與文件）
- 跑完記得：執行階段 → **中斷連線並刪除執行階段**